# Regressão linear múltipla

Julia Welter, Pedro Pacheco, Pedro K. Albuquerque e Tiago Rosa

Estudo da relação entre preço, tamanho do imóvel e quantidade de quartos.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.style.use('seaborn-v0_8-whitegrid')

try:
    from sklearn.linear_model import LinearRegression
    SKLEARN_OK = True
except Exception:
    SKLEARN_OK = False

In [ ]:
COLUNAS = ['tamanho', 'quartos', 'preco']

def carregar_base():
    csv_path = Path('dados/data.csv')
    mat_path = Path('dados/data.mat')

    if csv_path.exists():
        return pd.read_csv(csv_path, header=None, names=COLUNAS)

    if mat_path.exists():
        from scipy.io import loadmat
        mat = loadmat(mat_path)
        for valor in mat.values():
            if isinstance(valor, np.ndarray) and valor.ndim == 2 and valor.shape[1] >= 3:
                return pd.DataFrame(valor[:, :3], columns=COLUNAS)

    raise FileNotFoundError('Nao encontrei dados/data.csv nem dados/data.mat.')

def montar_X_y(base):
    x_base = base[['tamanho', 'quartos']].to_numpy(dtype=float)
    y = base['preco'].to_numpy(dtype=float).reshape(-1, 1)
    intercepto = np.ones((x_base.shape[0], 1))
    x = np.hstack((intercepto, x_base))
    return x_base, x, y

def regmultipla(X, y):
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float).reshape(-1, 1)
    return np.linalg.solve(X.T @ X, X.T @ y)

def calcular_theta(base):
    _, x, y = montar_X_y(base)
    return regmultipla(x, y)

def calcular_correlacoes(base):
    preco = base['preco'].to_numpy(dtype=float)
    corr_tamanho = float(np.corrcoef(base['tamanho'], preco)[0, 1])
    corr_quartos = float(np.corrcoef(base['quartos'], preco)[0, 1])
    return corr_tamanho, corr_quartos

## a) Carregamento dos dados

In [ ]:
base = carregar_base()
print(f'Total de exemplos: {len(base)}')
base.head()

## b) Análise estatística

In [ ]:
describe = base.describe()
media_preco = base['preco'].mean()
indice_menor_casa = base['tamanho'].idxmin()
preco_menor_casa = base.loc[indice_menor_casa, 'preco']
indice_mais_cara = base['preco'].idxmax()
quartos_mais_cara = int(base.loc[indice_mais_cara, 'quartos'])

print(f'Media de preco das casas: {media_preco:.2f}')
print(f'Preco da menor casa: {preco_menor_casa:.2f}')
print(f'Quartos da casa mais cara: {quartos_mais_cara}')
describe

## c) Montagem de X e y

In [ ]:
x_base, x, y = montar_X_y(base)
print(f'Shape de X_base: {x_base.shape}')
print(f'Shape de X: {x.shape}')
print(f'Shape de y: {y.shape}')

display(base[['tamanho', 'quartos']].head())
display(base.assign(intercepto=1)[['intercepto', 'tamanho', 'quartos']].head())
display(base[['preco']].head())

## d) Correlação e regressão linear simples

In [ ]:
tamanho = base['tamanho'].to_numpy(dtype=float)
quartos = base['quartos'].to_numpy(dtype=float)
preco = base['preco'].to_numpy(dtype=float)

corr_tamanho, corr_quartos = calcular_correlacoes(base)
ang_tamanho, lin_tamanho = np.polyfit(tamanho, preco, 1)
ang_quartos, lin_quartos = np.polyfit(quartos, preco, 1)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].scatter(tamanho, preco, edgecolors='blue', facecolors='none')
axes[0].plot(tamanho, lin_tamanho + ang_tamanho * tamanho, 'r-')
axes[0].set_title(f'Tamanho x Preco\ncorrelacao = {corr_tamanho:.4f}')
axes[0].set_xlabel('Tamanho')
axes[0].set_ylabel('Preco')

axes[1].scatter(quartos, preco, edgecolors='blue', facecolors='none')
axes[1].plot(quartos, lin_quartos + ang_quartos * quartos, 'r-')
axes[1].set_title(f'Quartos x Preco\ncorrelacao = {corr_quartos:.4f}')
axes[1].set_xlabel('Quartos')
axes[1].set_ylabel('Preco')

plt.tight_layout()
plt.show()

## e) Regressão múltipla

In [ ]:
theta = calcular_theta(base)
beta0, beta1, beta2 = theta.flatten()

fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(tamanho, quartos, preco, edgecolors='blue', facecolors='none', s=50)
ax.set_xlabel('Tamanho')
ax.set_ylabel('Quartos')
ax.set_zlabel('Preco')
plt.show()

print(f'beta0 (intercepto): {beta0:.2f}')
print(f'beta1 (tamanho):    {beta1:.6f}')
print(f'beta2 (quartos):    {beta2:.6f}')
print(f'Equacao: preco = {beta0:.2f} + ({beta1:.6f} * tamanho) + ({beta2:.6f} * quartos)')

## f) Linha de regressão no gráfico 3D

In [ ]:
y_reg = beta0 + beta1 * tamanho + beta2 * quartos

fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(tamanho, quartos, preco, edgecolors='blue', facecolors='none', s=50)
ax.plot(tamanho, quartos, y_reg, 'r-', linewidth=2)
ax.set_xlabel('Tamanho da casa')
ax.set_ylabel('Numero de quartos')
ax.set_zlabel('Preco da casa')
ax.view_init(elev=22, azim=45)
plt.show()

## g) Plano de regressão e correlações

In [ ]:
grade_tamanho = np.linspace(tamanho.min(), tamanho.max(), 20)
grade_quartos = np.linspace(quartos.min(), quartos.max(), 20)
malha_tamanho, malha_quartos = np.meshgrid(grade_tamanho, grade_quartos)
malha_preco = beta0 + beta1 * malha_tamanho + beta2 * malha_quartos

fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(tamanho, quartos, preco, edgecolors='blue', facecolors='none', s=50)
ax.plot_surface(malha_tamanho, malha_quartos, malha_preco, alpha=0.35, cmap='Reds')
ax.set_xlabel('Tamanho')
ax.set_ylabel('Quartos')
ax.set_zlabel('Preco')
ax.set_title(
    'Regressao Multipla\n'
    f'corr(tamanho, preco) = {corr_tamanho:.4f}\n'
    f'corr(quartos, preco) = {corr_quartos:.4f}'
)
plt.show()

## h) Previsões

In [ ]:
preco_3_quartos = float(np.array([1, 1650, 3]) @ theta.flatten())
preco_2_quartos = float(np.array([1, 1650, 2]) @ theta.flatten())
preco_4_quartos = float(np.array([1, 1650, 4]) @ theta.flatten())

print(f'3 quartos: {preco_3_quartos:.2f}')
print(f'2 quartos: {preco_2_quartos:.2f}')
print(f'4 quartos: {preco_4_quartos:.2f}')
print('Mantendo o tamanho fixo, a variacao segue o sinal do coeficiente associado a quartos.')

## i) Comparação com scikit-learn

In [ ]:
if SKLEARN_OK:
    modelo = LinearRegression()
    modelo.fit(x_base, y)
    theta_sklearn = np.array([
        modelo.intercept_.item(),
        modelo.coef_.flatten()[0],
        modelo.coef_.flatten()[1],
    ])
    print('Equacao normal:')
    print(theta.flatten())
    print('Scikit-learn:')
    print(theta_sklearn)
else:
    print('scikit-learn nao esta instalado neste ambiente.')